In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.autograd import Variable
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm  # Import tqdm for progress bar
from collections import deque

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load and normalize data

In [ ]:
# Load and normalize data
res_df = pd.read_excel("hourly_data.xlsx") #Moroccan data in hourly intervals
com_df = pd.read_excel("commercial_data.xlsx")
com_df['Commercial_load'] = com_df['Commercial_load'] + 250000 # offset to match Moroccan data ranges

loads_df = pd.DataFrame({
    'Residential_load':res_df['Total Power Consumption'][:192*3].values,
    'Commercial_load':com_df['Commercial_load'][:192*3].values
})

plt.figure(figsize=(10, 6))
plt.plot(loads_df['Residential_load'], label='Residential_load')
plt.plot(loads_df['Commercial_load'], label='Commercial_load')

# Customize the plot
plt.xlabel("Date")
plt.ylabel("Power consumed (W)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Normalize data
min_max_scaler = MinMaxScaler()
grad_scaler = torch.amp.GradScaler('cuda')
res_max = 700000
com_max = 600000
res_load_data = (loads_df['Residential_load'].values).reshape(-1, 1)
res_norm_load = min_max_scaler.fit_transform(res_load_data)

com_load_data = (loads_df['Commercial_load'].values).reshape(-1, 1)
com_norm_load = min_max_scaler.fit_transform(com_load_data)

util_res_load_data = (loads_df['Residential_load'].values/res_max).reshape(-1, 1)
util_res_norm_load = min_max_scaler.fit_transform(util_res_load_data)

util_com_load_data = (loads_df['Commercial_load'].values/com_max).reshape(-1, 1)
util_com_norm_load = min_max_scaler.fit_transform(util_com_load_data)


# Definition of a DDPG Agent

In [ ]:
# Actor network
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(Actor, self).__init__()
        self.fc1 = nn.Linear(state_dim, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_dim)
        self.tanh = nn.Tanh()
    
    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        x = self.tanh(self.fc3(x))  # Output between -1 and 1
        return (x + 1) / 2  # Scale to range 0 to 1

# Critic network
class Critic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(Critic, self).__init__()
        self.fc1 = nn.Linear(state_dim + action_dim, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 1)
    
    def forward(self, state, action):
        x = torch.cat([state, action], dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


In [ ]:
class DDPGAgent:
    def __init__(self, state_dim, action_dim, lr_actor=3e-4, lr_critic=3e-4, gamma=0.99, tau=0.005):
        self.actor = Actor(state_dim, action_dim).to(dev)
        self.critic = Critic(state_dim, action_dim).to(dev)
        self.target_actor = Actor(state_dim, action_dim).to(dev)
        self.target_critic = Critic(state_dim, action_dim).to(dev)
        
        self.target_actor.load_state_dict(self.actor.state_dict())
        self.target_critic.load_state_dict(self.critic.state_dict())
        
        # Use Adam optimizer for both the actor and the critic
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=lr_critic)
        self.gamma = gamma
        self.tau = tau
    
        # --- Track metrics ---
        self.last_actor_loss = 0.0
        self.last_critic_loss = 0.0
    
    def select_action(self, state):
        state = torch.tensor(state, dtype=torch.float32, device=dev).unsqueeze(0)
        with torch.no_grad():
            action = self.actor(state).cpu().numpy()
        return action[0]

    def update(self, replay_buffer, batch_size=64):
        if len(replay_buffer) < batch_size:
            return
        
        batch = [replay_buffer[i] for i in np.random.choice(len(replay_buffer), batch_size, replace=False)]
        states, actions, rewards, next_states = zip(*batch)
        
        states = torch.tensor(states, dtype=torch.float32, device=dev)
        actions = torch.tensor(actions, dtype=torch.float32, device=dev)
        rewards = torch.tensor(rewards, dtype=torch.float32, device=dev).unsqueeze(1)
        next_states = torch.tensor(next_states, dtype=torch.float32, device=dev)

        next_actions = self.target_actor(next_states)
        target_q = rewards + self.gamma * self.target_critic(next_states, next_actions).detach()
        
        # Get MSE loss
        with torch.amp.autocast('cuda'):
            q_values = self.critic(states, actions)
            critic_loss = F.mse_loss(q_values, target_q)

        self.critic_optimizer.zero_grad()
        grad_scaler.scale(critic_loss).backward()
        grad_scaler.step(self.critic_optimizer)
        grad_scaler.update()

        with torch.amp.autocast('cuda'):
            actor_loss = -self.critic(states, self.actor(states)).mean()

        self.actor_optimizer.zero_grad()
        grad_scaler.scale(actor_loss).backward()
        grad_scaler.step(self.actor_optimizer)
        grad_scaler.update()
        
        # Save metrics for logging
        self.last_actor_loss = actor_loss.item()
        self.last_critic_loss = critic_loss.item()
        
        for param, target_param in zip(self.actor.parameters(), self.target_actor.parameters()):
            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)
        
        for param, target_param in zip(self.critic.parameters(), self.target_critic.parameters()):
            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)


# Training the agent

In [ ]:
# Training
state_dim = 24  # 24-hour load states
action_dim = 24  # 24 actions per time step
agent = DDPGAgent(state_dim=state_dim, action_dim=action_dim)
replay_buffer = deque()
num_episodes = 15000
update_every = 50  # update every 50 steps
batch_size = 64

res_utils_diff = np.array(util_res_norm_load - util_com_norm_load)
res_utils_diff[res_utils_diff < 0] = 0 #replace negative entries with zeroes

# Arrays to store metrics per episode
rewards_per_episode = []
actor_losses = []
critic_losses = []

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for episode in tqdm(range(num_episodes), desc="Training Progress"):
        total_reward = 0  # Initialize total reward for the episode
        
        for i in range(len(res_norm_load) - 24):
            state = res_norm_load[i:i+24].flatten()  # 24-hour load as state
            action = agent.select_action(state)  # 24 separate actions
            reward = -np.abs(action - res_norm_load[i+1:i+25].flatten()).sum() # reward for peak-shaving and valley-filling
            # reward = -np.abs(action - res_utils_diff[i+1:i+25].flatten()).sum() # reward for inter-distribution networks load balancing
            # reward = -np.abs(action - res_norm_load[i+1:i+25].flatten()).sum() - (np.abs(action - res_utils_diff[i+1:i+25].flatten()).sum()) # reward for both
            total_reward += reward  # Accumulate reward
            
            next_state = res_norm_load[i+1:i+25].flatten() if i + 25 < len(res_norm_load) else state
            replay_buffer.append((state, action, reward, next_state))
            
            if i % update_every == 0:
                for _ in range(10):  # multiple updates per batch
                    agent.update(replay_buffer, batch_size)
        
        rewards_per_episode.append(total_reward)  # Store total reward for the episode
        actor_losses.append(agent.last_actor_loss)
        critic_losses.append(agent.last_critic_loss)
        
        if episode % 1500 == 0:
            print(f"Episode {episode}, Total Reward: {total_reward}")


In [ ]:
# Plot rewards

plt.plot(rewards_per_episode)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
# plt.title("Rewards per Episode")
plt.show()


In [ ]:
# Plot losses

plt.plot(actor_losses, label = "actor loss")
# plt.plot(critic_losses, label = "critic loss")
plt.xlabel("Episode")
plt.ylabel("Loss")
# plt.title("Rewards per Episode")
plt.legend()
plt.show()

In [ ]:
plt.plot(critic_losses, label = "critic loss")
# plt.plot(critic_losses, label = "critic loss")
plt.xlabel("Episode")
plt.ylabel("Loss")
# plt.title("Rewards per Episode")
plt.legend()
plt.show()

In [ ]:
def save_ddpg(agent, filename="ddpg_agent.pth"):
    torch.save({
        'actor_state_dict': agent.actor.state_dict(),
        'critic_state_dict': agent.critic.state_dict(),
        'actor_optimizer_state_dict': agent.actor_optimizer.state_dict(),
        'critic_optimizer_state_dict': agent.critic_optimizer.state_dict()
    }, filename)
    print(f"DDPG agent saved to {filename}")

def load_ddpg(agent, filename="ddpg_agent.pth"):
    checkpoint = torch.load(filename)
    agent.actor.load_state_dict(checkpoint['actor_state_dict'])
    agent.critic.load_state_dict(checkpoint['critic_state_dict'])
    agent.actor_optimizer.load_state_dict(checkpoint['actor_optimizer_state_dict'])
    agent.critic_optimizer.load_state_dict(checkpoint['critic_optimizer_state_dict'])
    print(f"DDPG agent loaded from {filename}")

In [ ]:
save_ddpg(agent)

In [ ]:
torch.backends.cudnn.benchmark = True

In [ ]:
state_dim = 24  # 24-hour load states
action_dim = 24  # 24 actions per time step
agent = DDPGAgent(state_dim=state_dim, action_dim=action_dim)

df = pd.read_excel("commercial_data.xlsx")
df = df[(192*3):(192*3)+24]
load_data = df['Commercial_load'].values.reshape(-1, 1)
scaler = MinMaxScaler()
norm_load = scaler.fit_transform(load_data)
state = norm_load.flatten()

load_ddpg(agent, filename='ddpg_agent.pth')
action = agent.select_action(state)  # 24 separate actions

In [ ]:
# Create the figure and primary axis
fig, ax1 = plt.subplots()

# Plot the first curve (left y-axis)
ax1.plot(df['Commercial_load'], 'b-', label='Commercial_load')
ax1.set_xlabel("Time (hours)")
ax1.set_ylabel("Total power consumption (kW)", color='b')
ax1.tick_params(axis='y', labelcolor='b')

# Create a second y-axis
ax2 = ax1.twinx()

# Plot the second curve (right y-axis)
ax2.plot(action, 'r--', label="Actions")
ax2.set_ylabel("Actions", color='r')
ax2.tick_params(axis='y', labelcolor='r')

# Show the plot
# plt.title("Two Curves with Different Y-Axis Values")
fig.tight_layout()
plt.show()

# plt.plot(action)
# plt.xlabel("Episode")
# plt.ylabel("Total Reward")
# plt.title("Rewards per Episode")
# plt.show()


Export stats to excel

In [ ]:
stats_df = pd.DataFrame({
    'DDPG rewards': rewards_per_episode,
    'DDPG actor loss': actor_losses,
    'DDPG critic loss': critic_losses
})
stats_df.to_excel("ddpg_stats.xlsx", index = False)
print("Data exported to ddpg_stats.xlsx")